In [77]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from dotenv import load_dotenv
from datetime import datetime
from email.message import EmailMessage
from IPython.display import display, Image, Markdown
from pathlib import Path
from typing import Literal
import base64
import os
import smtplib

from web_search_tool import FreeWebSearch

In [78]:
load_dotenv(override=True)

True

In [79]:
import os
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

PROVIDER = "gemini"  # "gemini", "groq", හෝ "openai" තෝරන්න

if PROVIDER == "gemini":
    set_tracing_disabled(True)
    external_client = AsyncOpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.getenv("GEMINI_API_KEY"),
        max_retries=5,
    )
    MODEL = OpenAIChatCompletionsModel(
        model="gemini-3.5-flash-lite",
        openai_client=external_client
    )

elif PROVIDER == "groq":
    set_tracing_disabled(True)
    external_client = AsyncOpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
        max_retries=5,
    )
    MODEL = OpenAIChatCompletionsModel(
        model="openai/gpt-oss-120b",
        openai_client=external_client
    )

else:
    # OpenAI පාවිච්චි කරනවා නම්
    MODEL = "gpt-4.1-mini"

In [80]:
MODEL_NAME = MODEL
HOW_MANY_SEARCHES = 2

IMAGE_MODEL = "gpt-image-2.5-flare"
IMAGE_QUALITY = "low"      # About $0.005 per image
IMAGE_SIZE = "1024x1536"   # Portrait

MAX_IMAGES = 4

EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

POSTERS_DIR = Path("creative_agency_posters")

if PROVIDER == "openai": client = AsyncOpenAI()

In [81]:
# Agent 1 -> Planning Agent



In [82]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search query to use for this web search.")

In [83]:
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform, with reasoning for each search.")

In [84]:
INSTRUCTIONS = f"""
You are an expert research strategist. Given a user's request, design a comprehensive, high-impact web search plan that gathers all the context needed to produce an exceptional result.

Guidelines:
1. **Diverse Angles:** Break the topic down into distinct, non-overlapping dimensions (e.g., current trends, target audience insights, competitor examples, key facts/statistics, and visual or creative inspiration).
2. **Search-Engine Optimized Queries:** Write specific, targeted search queries that yield rich, actionable results. Avoid vague, generic, or overly long conversational queries.
3. **Zero Redundancy:** Ensure every query serves a unique purpose and does not duplicate another search.
4. **Clear Reasoning:** For each query, briefly explain the exact gap it fills and how it directly supports the user's goal.
5. **Strict Count:** Output exactly {HOW_MANY_SEARCHES} web searches.
6. **JSON Format:** Return the output strictly as valid JSON matching the schema.
"""

In [85]:
planing_agent = Agent(name="Kavindu", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [86]:
result = await Runner.run(planing_agent, "Create viral Instagram posters for Ocean Breeze Hotel's new rooftop dining experience")
print(result.final_output)

searches=[WebSearchItem(reason='Identify current viral Instagram design trends, aesthetic styles, and engaging visual formats specifically for luxury rooftop dining and hospitality marketing.', query='viral instagram poster trends luxury rooftop dining hotel marketing'), WebSearchItem(reason='Gather effective copywriting hooks, call-to-action strategies, and engagement tactics used by high-end hotels for rooftop restaurant launches.', query='rooftop restaurant launch instagram caption hooks copywriting strategy')]


In [87]:
# Agent 2 -> Web Search Agent

In [88]:
INSTRUCTIONS = """
You are an expert web research analyst. When given a search term, search the web and synthesize the most valuable, up-to-date findings into a tight, high-signal briefing.

Rules:
1. **Prioritize Concrete Details:** Extract specific facts, numbers, trends, brand details, or actionable insights—skip generic filler and fluff.
2. **Focus on Relevance:** Filter out noise, ads, and outdated info. Only include details that directly answer the search intent.
3. **Strict Length:** Write 2–3 tight paragraphs, strictly under 300 words.
4. **Clean Output:** Output ONLY the summary. Do not include intros ("Here is what I found..."), meta-commentary, or sign-offs.
"""

In [89]:
## web search eka salli kapenawa, mekedi use kare open ai lage tool ekak hadala tiyena, e nisa eka parakata sathaya gane kapenawa
#  search_agent = Agent(name="Tachini", instructions=INSTRUCTIONS, model=MODEL_NAME, tools=[WebSearchTool()], model_settings=ModelSettings(tool_choice="required"))

In [90]:
# result = await Runner.run(search_agent, "Create viral Instagram posters for Ocean Breeze Hotel's new rooftop dining experience")
# print(result.final_output)

In [91]:
## me ap hadagaththa fre web search tool eka call karana eka
from web_search_tool import FreeWebSearch

# Object එක හදාගන්න (ඕන නම් max_results වෙනස් කරගන්න පුළුවන්)
searcher = FreeWebSearch(max_results=5)

# Agent ට tool එකක් විදිහට දෙන්න
search_agent = Agent(
    name="Tachini",
    instructions=INSTRUCTIONS,
    model=MODEL_NAME,
    tools=[searcher.as_agent_tool()],
    model_settings=ModelSettings(tool_choice="auto")
)

In [92]:
result = await Runner.run(search_agent, "Create viral Instagram posters for Ocean Breeze Hotel's new rooftop dining experience")
print(result.final_output)

**Poster 1: The Golden Hour Vibe**
* **Visual:** A high-contrast, sun-drenched photo of a cocktail with condensation, held up against a blazing pastel sunset over the ocean. The rooftop lounge is softly silhouetted in the background with fairy lights glowing.
* **Headline:** "Sip Above the Sea."
* **Sub-text:** *Ocean Breeze Rooftop Dining is Now Open.* Elevate your evenings with panoramic ocean views, craft cocktails, and coastal-inspired small plates. 
* **Design/Sticker Layer:** Add an interactive IG countdown sticker: "First look this Friday!" Use warm, aesthetic typography (like a blend of serif and clean sans-serif) and earthy tones to maximize saves and shares.

**Poster 2: The FOMO / Late-Night Appeal**
* **Visual:** A sleek, moody drone shot or wide-angle photo of the rooftop at night, showing ambient lighting, chic velvet seating, and a bustling vibe under the stars. 
* **Headline:** "Where the Sky Meets the Shore."
* **Sub-text:** Your new favorite weekend spot just dropped.

In [93]:
# Agent 3 -> Script Writing Agent

In [94]:
INSTRUCTIONS = """
You are a senior creative writer who makes posters that go viral on social media.
You will receive the user's request along with supporting research.
Use both to write exactly 2 poster scripts, each with a different angle.

For each script give:

- Poster number: 1 or 2
- Hook: the idea that makes people stop scrolling and share
- Text on the poster: a headline of at most 6 words, plus any other short text
- Visuals: the subject, style, colours and layout

Use only facts from the user's request and the research.
Never invent prices, dates, offers, locations or contact details.
"""

In [95]:
script_agent = Agent(name="Chamod", instructions=INSTRUCTIONS, model=MODEL_NAME,)

In [96]:
result = await Runner.run(script_agent, "Create viral Instagram posters for Ocean Breeze Hotel's new rooftop dining experience")
print(result.final_output)

### Poster 1

* **Poster number:** 1
* **Hook:** Appeal to the high-status, aesthetic-driven crowd looking for the ultimate sunset photo spot and luxury evening vibe.
* **Text on the poster:** 
  * Headline: Elevate your dining experience. (4 words)
  * Subtext: Ocean Breeze Hotel rooftop dining is now open.
* **Visuals:** 
  * Subject: A chic rooftop terrace at golden hour, featuring sleek modern dining tables, glowing ambient lighting, and guests enjoying cocktails with a sweeping ocean horizon view.
  * Style: Editorial, high-fashion lifestyle photography, clean and aspirational.
  * Colours: Warm sunset gold, deep ocean navy, and crisp white accents.
  * Layout: Full-bleed image with minimalist white typography centered in the upper third, leaving the breathtaking view and atmosphere to dominate the frame.

***

### Poster 2

* **Poster number:** 2
* **Hook:** Tap into sensory cravings and experiential travel—combining incredible food with unmatched open-air sea views.
* **Text on 

In [97]:
# Agent 4 -> Image Genaration Agent

In [98]:
def show_error(context, error):
    """Tell the agent what really went wrong, instead of the SDK's generic 'try again' message."""
    return f"The tool failed: {error}"

In [99]:
import asyncio
import requests
from urllib.parse import quote

@function_tool(failure_error_function=show_error)
async def create_poster(poster_number: Literal[1, 2], prompt: str) -> str:
    """
    Generate one poster image and save it

    Args:
        poster_number: Which poster this is, 1 or 2
        prompt: A detailed description of the poster, with any text to show in double quotes
    """
    global images_used

    if images_used >= MAX_IMAGES:
        raise RuntimeError(f"The budget of {MAX_IMAGES} images is used up. Do not generate any more.")

    print(f"Creating poster {poster_number}...")
    path = run_folder / f"poster_{poster_number}.png"

    if PROVIDER == "openai":
        # Production (OpenAI)
        response = await client.images.generate(
            model=IMAGE_MODEL,
            prompt=prompt,
            size=IMAGE_SIZE,
            quality=IMAGE_QUALITY
        )
        path.write_bytes(base64.b64decode(response.data[0].b64_json))
    else:
        # Free Testing: nologo=true අයින් කරලා try කරනවා
        short_prompt = prompt[:400]
        encoded_prompt = quote(short_prompt)
        url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=768&height=1024"

        saved = False
        for attempt in range(2):
            try:
                res = requests.get(url, timeout=60)
                res.raise_for_status()
                path.write_bytes(res.content)
                saved = True
                break
            except Exception as e:
                print(f"Poster {poster_number} attempt {attempt + 1} failed: {e}")
                await asyncio.sleep(3)

        # Pollinations 402/fail වුණොත් test workflow එක නවතින්නේ නැති වෙන්න fallback image එකක් ගන්නවා
        if not saved:
            print(f"Using fallback test image for poster {poster_number}...")
            fallback = requests.get(f"https://picsum.photos/768/1024?random={poster_number}", timeout=30)
            fallback.raise_for_status()
            path.write_bytes(fallback.content)

    images_used += 1
    return f"Saved {path.name}"

In [100]:
INSTRUCTIONS = """
You will receive 2 poster scripts. Call create_poster once for each script.

Write a detailed prompt covering the subject, style, colours and layout.
Put the exact text to show in double quotes, and do not change it.

Reply with the file names you created.
"""

In [101]:
create_agent = Agent(name="Nadeesha", instructions=INSTRUCTIONS, model=MODEL_NAME, tools=[create_poster], model_settings=ModelSettings(tool_choice="required"))

In [102]:
# Agent 5 -> Email sending Agent

In [103]:
@function_tool(failure_error_function=show_error)
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send an email to us with the posters from this run attached

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    posters = sorted(run_folder.glob("*.png"))

    if not posters:
        raise RuntimeError("There are no posters to send yet.")

    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    for path in posters:
        msg.add_attachment(path.read_bytes(), maintype="image", subtype="png", filename=path.name)

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

    return f"Email sent with {len(posters)} posters attached"

In [104]:
INSTRUCTIONS = """
You will receive the user's request and 2 poster scripts.
Write a short, clean HTML email with a suitable subject line. Explain the hook of each poster.
The posters are attached automatically, so do not add images or links to them.
Then use your tool to send the email.
"""

In [105]:
email_agent = Agent(name="Ranaweera", instructions=INSTRUCTIONS, model=MODEL_NAME, tools=[send_email_tool], model_settings=ModelSettings(tool_choice="required"))

In [106]:
## Now Orcastrate Agent   use LLM method

In [107]:
planner_tool = planing_agent.as_tool(
    tool_name="search_planner",
    tool_description="Plan the web searches for the user's request. In the input, give the full request.",
)

search_tool = search_agent.as_tool(
    tool_name="web_searcher",
    tool_description="Search the web for one search term and get a short summary. In the input, give the search term and the reason for searching.",
)

script_writer_tool = script_agent.as_tool(
    tool_name="script_writer",
    tool_description="Write 2 viral poster scripts. In the input, give the full request and all of the search summaries.",
)

creative_tool = create_agent.as_tool(
    tool_name="poster_creator",
    tool_description="Create a poster image from each script. In the input, give both full scripts.",
)

email_tool = email_agent.as_tool(
    tool_name="poster_emailer",
    tool_description="Email the finished posters to us. In the input, give the full request and both full scripts.",
)

tools = [planner_tool, search_tool, script_writer_tool, creative_tool, email_tool]

In [108]:
INSTRUCTIONS = """
You are the Creative Director of a creative agency.
You turn the user's request into 2 viral posters and email them, using your tools.
You never write scripts or make posters yourself.

Follow these steps:

1. Plan: Use search_planner to get the list of searches for the request.

2. Search: Use web_searcher once for each search in the plan.

3. Write: Use script_writer. Give it the request and all of the search summaries.

4. Create: Use poster_creator. Give it both scripts.

5. Send the Email: Use poster_emailer once, after the posters are created.
Give it the request and both scripts.

6. Finish: Say which posters were emailed, and what went wrong if anything did.

Always pass the full text to your tools, not a summary.
"""

In [109]:
creative_director_agent = Agent(name="Sajani", instructions=INSTRUCTIONS, model=MODEL_NAME, tools=tools, model_settings=ModelSettings(parallel_tool_calls=False))

In [110]:
# A new folder for each run
run_folder = POSTERS_DIR / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_folder.mkdir(parents=True)
images_used = 0

# max_turns: every tool call here is one turn, so a normal run uses all of the default 10.
# 20 leaves room for a retry.
with trace("Creative agency"):
    result = await Runner.run(
        creative_director_agent,
        "Create viral Instagram posters for Ocean Breeze Hotel's new rooftop dining experience",
        max_turns=20
    )

display(Markdown(result.final_output))

Creating poster 1...
Creating poster 2...
Poster 2 attempt 1 failed: 402 Client Error: Payment Required for url: https://image.pollinations.ai/prompt/A%20mixed%20media%20aesthetic%20poster%20combining%20high-fashion%20magazine%20minimalism%20with%20a%20torn-paper%20edge%20overlay%20effect.%20Rich%20sunset%20gradients%20of%20burnt%20orange%2C%20soft%20rose%2C%20and%20deep%20midnight%20navy.%20An%20asymmetrical%20layout%20with%20the%20serif%20headline%20placed%20vertically%20down%20the%20left%20margin%20reading%20%22The%20Best%20Sunset%20In%20Town%22%2C%20overlapping%20a%20textured%20golden%20hour%20photo%20on%20the%20right.%20The%20photo%20is%20a%20close-up%20of%20a%20vibrant%2C?width=768&height=1024
Poster 2 attempt 2 failed: 402 Client Error: Payment Required for url: https://image.pollinations.ai/prompt/A%20mixed%20media%20aesthetic%20poster%20combining%20high-fashion%20magazine%20minimalism%20with%20a%20torn-paper%20edge%20overlay%20effect.%20Rich%20sunset%20gradients%20of%20burnt%2

The two viral Instagram posters for the Ocean Breeze Hotel's new rooftop dining experience have been successfully created and emailed to you! 

* **Poster 1 ("Meet Me Above The Clouds"):** Features a modern editorial layout with a cinematic twilight skyline silhouette, deep blues, and warm amber tones.
* **Poster 2 ("The Best Sunset In Town"):** Features an asymmetrical mixed-media design highlighting a golden hour cocktail against a blurred skyline bokeh and rich sunset gradients.

Everything was completed smoothly with no errors.

In [111]:
## Ocr part by code (second methodd)

In [112]:
import asyncio

async def run_creative_agency(request: str) -> str:
    # 1. Plan
    plan_result = await Runner.run(planing_agent, request)
    search_plan = plan_result.final_output
    await asyncio.sleep(5)

    # 2. Search
    search_summaries = []
    for item in search_plan.searches:
        search_input = f"Search term: {item.query}\nReason: {item.reason}"
        search_res = await Runner.run(search_agent, search_input)
        search_summaries.append(f"Query: {item.query}\nSummary: {search_res.final_output}")
        await asyncio.sleep(5)
    all_research = "\n\n---\n\n".join(search_summaries)

    # 3. Write
    script_input = f"User Request:\n{request}\n\nSupporting Research:\n{all_research}"
    script_result = await Runner.run(script_agent, script_input)
    scripts = script_result.final_output
    await asyncio.sleep(5)

    # 4. Create
    await Runner.run(create_agent, scripts)
    await asyncio.sleep(5)

    # 5. Send the Email
    email_input = f"User Request:\n{request}\n\nPoster Scripts:\n{scripts}"
    email_result = await Runner.run(email_agent, email_input)

    # 6. Finish
    return email_result.final_output

In [113]:
# A new folder for each run
run_folder = POSTERS_DIR / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_folder.mkdir(parents=True, exist_ok=True)
images_used = 0

with trace("Creative agency - Code Orchestration"):
    final_output = await run_creative_agency(
        "Create viral Instagram posters for Ocean Breeze Hotel's new rooftop dining experience"
    )

display(Markdown(final_output))

Creating poster 1...
Creating poster 2...
Poster 2 attempt 1 failed: 402 Client Error: Payment Required for url: https://image.pollinations.ai/prompt/A%20clean%20architectural%20vertical%20poster%20with%20warm%20golden-hour%20framing%20and%20cinematic%20nostalgia.%20In%20the%20upper%20third%2C%20elegant%20typography%20reads%20%27WHERE%20THE%20OCEAN%20BREEZE%20MEETS%20HIGH%20CUISINE%27.%20Below%20it%2C%20sub-text%20reading%20%27Now%20Open.%20Secure%20your%20table%20at%20the%20edge%20of%20the%20sky.%27.%20The%20visual%20is%20a%20close-up%2C%20tactile%20shot%20of%20artisanal%20coastal-inspired%20small%20plates%20and%20tableside%20mixology%20glowing%20in%20warm%20golden-hour%20li?width=768&height=1024
Poster 2 attempt 2 failed: 402 Client Error: Payment Required for url: https://image.pollinations.ai/prompt/A%20clean%20architectural%20vertical%20poster%20with%20warm%20golden-hour%20framing%20and%20cinematic%20nostalgia.%20In%20the%20upper%20third%2C%20elegant%20typography%20reads%20%27WHER

I have sent an email containing a summary and the hooks for both viral Instagram poster concepts for the Ocean Breeze Hotel rooftop dining experience. The posters are attached to the email for your review!

In [114]:
import smtplib

print("Posters in folder:", list(run_folder.glob("*.png")))
print("EMAIL_ADDRESS:", EMAIL_ADDRESS)
print("EMAIL_SMTP_SERVER:", EMAIL_SMTP_SERVER)
print("PASSWORD loaded?:", bool(EMAIL_APP_PASSWORD), "| Length:", len(EMAIL_APP_PASSWORD or ""))

try:
    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587, timeout=15) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        print("SMTP Login OK! Email වැඩ.")
except Exception as e:
    print("SMTP Error:", repr(e))

Posters in folder: [WindowsPath('creative_agency_posters/2026-11-20_01-12-33/poster_1.png'), WindowsPath('creative_agency_posters/2026-11-20_01-12-33/poster_2.png')]
EMAIL_ADDRESS: kchamod1124@gmail.com
EMAIL_SMTP_SERVER: smtp.gmail.com
PASSWORD loaded?: True | Length: 16
SMTP Login OK! Email වැඩ.
